# NSP Intent JSON Generation - Fine-Tuned Qwen3.5-9B Demo

This notebook demonstrates the fine-tuned Qwen3.5-9B model that converts **natural language instructions** into **NSP-ready intent JSON configurations**.

We validate the model against **real operational data from Sarvesh's NSP deployments**, then show additional examples across all supported intent types.

## 1. Load the Fine-Tuned Model

In [1]:
import json
import re
import torch
import sys
import os
from transformers import AutoModelForCausalLM, AutoTokenizer, StoppingCriteria, StoppingCriteriaList
from peft import PeftModel
from IPython.display import display, HTML, Markdown

sys.path.insert(0, os.path.join(os.getcwd(), 'inference'))
from merge_fill_values import merge_fill_values

MODEL_NAME = "Qwen/Qwen3.5-9B"
ADAPTER_DIR = "output/qwen3-nsp-intent-adapter"

SYSTEM_PROMPT = (
    "You are an NSP (Network Services Platform) intent configuration assistant. "
    "Given a natural language description of a network service, output a JSON object with three fields: "
    "'intent_type' (one of: epipe, tunnel, vprn), "
    "'template_name' (the NSP template to use), "
    "and 'fill_values' (a flat dictionary of field paths and their values that should be filled into the intent template). "
    "Only include fields that differ from template defaults. "
    "Use dot notation for nested paths and [N] for array indices. "
    "Output only valid JSON, no explanations."
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", trust_remote_code=True, torch_dtype=torch.bfloat16
)

print("Loading LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()
print("Model loaded successfully!")

/home/nextron/nsp_intent_ft/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading tokenizer...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model...


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

...

Loading weights:  94%|█████████▎| 400/427 [00:02<00:00, 149.61it/s]

Loading weights:  98%|█████████▊| 417/427 [00:02<00:00, 153.83it/s]

Loading weights: 100%|██████████| 427/427 [00:02<00:00, 145.77it/s]

Loading LoRA adapter...


Model loaded successfully!


## 2. Helper Functions

In [2]:
class JsonStoppingCriteria(StoppingCriteria):
    def __init__(self, tokenizer, start_len):
        self.tokenizer = tokenizer
        self.start_len = start_len
        self.brace_count = 0
        self.started = False

    def __call__(self, input_ids, scores, **kwargs):
        new_tokens = input_ids[0][self.start_len:].tolist()
        text = self.tokenizer.decode(new_tokens, skip_special_tokens=True)
        for char in text:
            if char == '{':
                self.brace_count += 1
                self.started = True
            elif char == '}':
                self.brace_count -= 1
        return self.started and self.brace_count <= 0


def extract_json(text):
    text = text.strip()
    m = re.search(r'(\{.*\})', text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            pass
    return None


def predict(instruction, max_new_tokens=1024, temperature=0.1):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": instruction},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    stopping = StoppingCriteriaList([JsonStoppingCriteria(tokenizer, input_len)])

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=temperature,
            top_p=0.95, eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id, stopping_criteria=stopping,
        )

    generated = outputs[0][input_len:]
    return tokenizer.decode(generated, skip_special_tokens=True)


def full_pipeline(instruction):
    """Run full pipeline: instruction -> fill_values -> merged API-ready JSON."""
    raw = predict(instruction)
    parsed = extract_json(raw)
    if parsed is None:
        return None, None
    intent_type = parsed.get("intent_type")
    fill_values = parsed.get("fill_values", {})
    merged = merge_fill_values(intent_type, fill_values)
    return parsed, merged


def show_comparison(title, instruction, ground_truth_path, ground_truth_json=None):
    """Run prediction and show side-by-side comparison with ground truth."""
    display(Markdown(f"### {title}"))
    display(Markdown(f"**User Instruction:**\n> {instruction}"))

    fill_values, api_json = full_pipeline(instruction)

    if ground_truth_json is None:
        with open(ground_truth_path) as f:
            ground_truth_json = json.load(f)

    model_str = json.dumps(api_json, indent=2)
    gt_str = json.dumps(ground_truth_json, indent=2)
    match = model_str == gt_str

    html = f"""
    <div style="display: flex; gap: 20px; margin: 10px 0;">
        <div style="flex: 1; border: 2px solid {'#28a745' if match else '#dc3545'}; border-radius: 8px; padding: 10px;">
            <h4 style="color: {'#28a745' if match else '#dc3545'};">Model Output {'\u2705' if match else '\u274c'}</h4>
            <pre style="font-size: 11px; max-height: 400px; overflow-y: auto;">{model_str}</pre>
        </div>
        <div style="flex: 1; border: 2px solid #007bff; border-radius: 8px; padding: 10px;">
            <h4 style="color: #007bff;">Ground Truth (Sarvesh's Real Data)</h4>
            <pre style="font-size: 11px; max-height: 400px; overflow-y: auto;">{gt_str}</pre>
        </div>
    </div>
    <p><strong>Result: {'EXACT MATCH' if match else 'MISMATCH'}</strong></p>
    """
    display(HTML(html))
    return fill_values, api_json


def show_example(title, instruction):
    """Run prediction and display result (no ground truth comparison)."""
    display(Markdown(f"### {title}"))
    display(Markdown(f"**User Instruction:**\n> {instruction}"))

    fill_values, api_json = full_pipeline(instruction)

    display(Markdown("**Step 1: Model extracts fill-values:**"))
    print(json.dumps(fill_values, indent=2))

    display(Markdown("**Step 2: Merged into API-ready JSON:**"))
    print(json.dumps(api_json, indent=2))
    return fill_values, api_json

print("Helper functions ready.")

Helper functions ready.


---
## 3. Validation Against Sarvesh's Real Operational Data

These are the **exact same instructions and configurations** that colleague Sarvesh used to deploy real services on the NSP system. We compare our model's output against the actual JSON payloads from his operational workflow.

### 3.1 MPLS Tunnel (from SarveshIntent.txt)

In [3]:
# Ground truth: the exact JSON from Sarvesh's real cURL command
tunnel_ground_truth = {
  "nsp-tunnel-intent:intent": [
    {
      "source-ne-id": "192.168.0.16",
      "sdp-id": "1637",
      "intent-type": "tunnel",
      "intent-type-version": "2",
      "olc-state": "deployed",
      "template-name": "MPLSTunnelsWithBGP",
      "intent-specific-data": {
        "tunnel:tunnel": {
          "destination-ne-id": "192.168.0.37",
          "admin-state": "unlocked",
          "transport-type": "mpls",
          "signaling": "tldp",
          "mpls": {
            "mixed-lsp-mode": False,
            "enable-ldp": False,
            "enable-bgp-tunnel": True,
            "sr-isis": False,
            "sr-ospf": False,
            "lsp": ""
          },
          "hello-parameters": {"keep-alive-enabled": False},
          "name": "SDP-from-C2U16-to-C2U37"
        }
      }
    }
  ]
}

show_comparison(
    "Test 1: MPLS Tunnel - Sarvesh's Real Deployment",
    "Create an MPLS tunnel from 192.168.0.16 to 192.168.0.37 with SDP ID 1637. "
    "Name it 'SDP-from-C2U16-to-C2U37'. Use BGP signaling with TLDP.",
    ground_truth_path=None,
    ground_truth_json=tunnel_ground_truth
)

### Test 1: MPLS Tunnel - Sarvesh's Real Deployment

**User Instruction:**
> Create an MPLS tunnel from 192.168.0.16 to 192.168.0.37 with SDP ID 1637. Name it 'SDP-from-C2U16-to-C2U37'. Use BGP signaling with TLDP.

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


({'intent_type': 'tunnel',
  'template_name': 'MPLSTunnelsWithBGP',
  'fill_values': {'source-ne-id': '192.168.0.16',
   'sdp-id': '1637',
   'destination-ne-id': '192.168.0.37',
   'name': 'SDP-from-C2U16-to-C2U37'}},
 {'nsp-tunnel-intent:intent': [{'source-ne-id': '192.168.0.16',
    'sdp-id': '1637',
    'intent-type': 'tunnel',
    'intent-type-version': '2',
    'olc-state': 'deployed',
    'template-name': 'MPLSTunnelsWithBGP',
    'intent-specific-data': {'tunnel:tunnel': {'destination-ne-id': '192.168.0.37',
      'admin-state': 'unlocked',
      'transport-type': 'mpls',
      'signaling': 'tldp',
      'mpls': {'mixed-lsp-mode': False,
       'enable-ldp': False,
       'enable-bgp-tunnel': True,
       'sr-isis': False,
       'sr-ospf': False,
       'lsp': ''},
      'hello-parameters': {'keep-alive-enabled': False},
      'name': 'SDP-from-C2U16-to-C2U37'}}}]})

### 3.2 ePIPE Service (from SarveshIntent.txt)

In [4]:
# Ground truth: Sarvesh's real epipe JSON
epipe_ground_truth_path = "../Documents/git/LLM_Hugging/Intents_Sarvesh/epipe_sarvesh.json"

show_comparison(
    "Test 2: ePIPE Service - Sarvesh's Real Deployment",
    "Create an E-Pipe service named 'Epipe-VLAN-1001-nvlink-C2U16-to-a5000dual-C2U35' "
    "for customer 10 (NE service ID 2001). Connect device 192.168.0.37 on port 1/2/c4/1 "
    "to device 192.168.0.16 on port 1/2/c5/1 using VLAN 1001. MTU is 1492. "
    "Use SDP 3716 and 1637.",
    ground_truth_path=epipe_ground_truth_path
)

### Test 2: ePIPE Service - Sarvesh's Real Deployment

**User Instruction:**
> Create an E-Pipe service named 'Epipe-VLAN-1001-nvlink-C2U16-to-a5000dual-C2U35' for customer 10 (NE service ID 2001). Connect device 192.168.0.37 on port 1/2/c4/1 to device 192.168.0.16 on port 1/2/c5/1 using VLAN 1001. MTU is 1492. Use SDP 3716 and 1637.

({'intent_type': 'epipe',
  'template_name': 'ePIPE-Service-Using-SDP',
  'fill_values': {'service-name': 'Epipe-VLAN-1001-nvlink-C2U16-to-a5000dual-C2U35',
   'customer-id': 10,
   'ne-service-id': 2001,
   'mtu': 1492,
   'site-a.device-id': '192.168.0.37',
   'site-a.endpoint[0].port-id': '1/2/c4/1',
   'site-a.endpoint[0].outer-vlan-tag': 1001,
   'site-b.device-id': '192.168.0.16',
   'site-b.endpoint[0].port-id': '1/2/c5/1',
   'site-b.endpoint[0].outer-vlan-tag': 1001,
   'sdp[0].sdp-id': '3716',
   'sdp[0].source-device-id': '192.168.0.37',
   'sdp[0].destination-device-id': '192.168.0.16',
   'sdp[1].sdp-id': '1637',
   'sdp[1].source-device-id': '192.168.0.16',
   'sdp[1].destination-device-id': '192.168.0.37'}},
 {'nsp-service-intent:intent': [{'intent-specific-data': {'epipe:epipe': {'admin-state': 'unlocked',
      'customer-id': 10,
      'mtu': 1492,
      'ne-service-id': 2001,
      'sdp-details': {'sdp': [{'admin-state': 'unlocked',
         'destination-device-id': '

---
## 4. Additional Examples - Different Intent Types & Styles

The following examples demonstrate the model's ability to handle:
- Different intent types (tunnel, epipe, VPRN)
- Different instruction styles (formal, conversational, terse)
- Complex multi-site VPRN configurations

### 4.1 Tunnel - Conversational Style

In [5]:
show_example(
    "Tunnel - Conversational Style",
    "I need an SDP tunnel between routers 10.0.1.1 and 10.0.2.2. "
    "Use BGP signaling with SDP ID 1122. Name it 'SDP-from-R1-to-R2'."
)

### Tunnel - Conversational Style

**User Instruction:**
> I need an SDP tunnel between routers 10.0.1.1 and 10.0.2.2. Use BGP signaling with SDP ID 1122. Name it 'SDP-from-R1-to-R2'.

**Step 1: Model extracts fill-values:**

{
  "intent_type": "tunnel",
  "template_name": "MPLSTunnelsWithBGP",
  "fill_values": {
    "source-ne-id": "10.0.1.1",
    "sdp-id": "1122",
    "destination-ne-id": "10.0.2.2",
    "name": "SDP-from-R1-to-R2"
  }
}


**Step 2: Merged into API-ready JSON:**

{
  "nsp-tunnel-intent:intent": [
    {
      "source-ne-id": "10.0.1.1",
      "sdp-id": "1122",
      "intent-type": "tunnel",
      "intent-type-version": "2",
      "olc-state": "deployed",
      "template-name": "MPLSTunnelsWithBGP",
      "intent-specific-data": {
        "tunnel:tunnel": {
          "destination-ne-id": "10.0.2.2",
          "admin-state": "unlocked",
          "transport-type": "mpls",
          "signaling": "tldp",
          "mpls": {
            "mixed-lsp-mode": false,
            "enable-ldp": false,
            "enable-bgp-tunnel": true,
            "sr-isis": false,
            "sr-ospf": false,
            "lsp": ""
          },
          "hello-parameters": {
            "keep-alive-enabled": false
          },
          "name": "SDP-from-R1-to-R2"
        }
      }
    }
  ]
}


({'intent_type': 'tunnel',
  'template_name': 'MPLSTunnelsWithBGP',
  'fill_values': {'source-ne-id': '10.0.1.1',
   'sdp-id': '1122',
   'destination-ne-id': '10.0.2.2',
   'name': 'SDP-from-R1-to-R2'}},
 {'nsp-tunnel-intent:intent': [{'source-ne-id': '10.0.1.1',
    'sdp-id': '1122',
    'intent-type': 'tunnel',
    'intent-type-version': '2',
    'olc-state': 'deployed',
    'template-name': 'MPLSTunnelsWithBGP',
    'intent-specific-data': {'tunnel:tunnel': {'destination-ne-id': '10.0.2.2',
      'admin-state': 'unlocked',
      'transport-type': 'mpls',
      'signaling': 'tldp',
      'mpls': {'mixed-lsp-mode': False,
       'enable-ldp': False,
       'enable-bgp-tunnel': True,
       'sr-isis': False,
       'sr-ospf': False,
       'lsp': ''},
      'hello-parameters': {'keep-alive-enabled': False},
      'name': 'SDP-from-R1-to-R2'}}}]})

### 4.2 ePIPE - Terse/Operational Style

In [6]:
show_example(
    "ePIPE - Terse/Operational Style",
    "Deploy epipe: Epipe-VLAN-200-DC-East-to-DC-West, cust=500, svc-id=3001, mtu=9000, "
    "10.10.1.1:1/1/c3/1 <-> 10.10.2.2:1/2/c6/1, VLAN 200, SDP 1122/2211"
)

### ePIPE - Terse/Operational Style

**User Instruction:**
> Deploy epipe: Epipe-VLAN-200-DC-East-to-DC-West, cust=500, svc-id=3001, mtu=9000, 10.10.1.1:1/1/c3/1 <-> 10.10.2.2:1/2/c6/1, VLAN 200, SDP 1122/2211

**Step 1: Model extracts fill-values:**

{
  "intent_type": "epipe",
  "template_name": "ePIPE-Service-Using-SDP",
  "fill_values": {
    "service-name": "Epipe-VLAN-200-DC-East-to-DC-West",
    "customer-id": 500,
    "ne-service-id": 3001,
    "mtu": 9000,
    "site-a.device-id": "10.10.1.1",
    "site-a.endpoint[0].port-id": "1/1/c3/1",
    "site-a.endpoint[0].outer-vlan-tag": 200,
    "site-b.device-id": "10.10.2.2",
    "site-b.endpoint[0].port-id": "1/2/c6/1",
    "site-b.endpoint[0].outer-vlan-tag": 200,
    "sdp[0].sdp-id": "1122",
    "sdp[0].source-device-id": "10.10.1.1",
    "sdp[0].destination-device-id": "10.10.2.2",
    "sdp[1].sdp-id": "2211",
    "sdp[1].source-device-id": "10.10.2.2",
    "sdp[1].destination-device-id": "10.10.1.1"
  }
}


**Step 2: Merged into API-ready JSON:**

{
  "nsp-service-intent:intent": [
    {
      "intent-specific-data": {
        "epipe:epipe": {
          "admin-state": "unlocked",
          "customer-id": 500,
          "mtu": 9000,
          "ne-service-id": 3001,
          "sdp-details": {
            "sdp": [
              {
                "admin-state": "unlocked",
                "destination-device-id": "10.10.2.2",
                "override-vc-id": false,
                "sdp-id": "1122",
                "source-device-id": "10.10.1.1",
                "vc-type": "ether"
              },
              {
                "admin-state": "unlocked",
                "destination-device-id": "10.10.1.1",
                "override-vc-id": false,
                "sdp-id": "2211",
                "source-device-id": "10.10.2.2",
                "vc-type": "ether"
              }
            ]
          },
          "site-a": {
            "device-id": "10.10.1.1",
            "endpoint": [
              {
                "admin-st

({'intent_type': 'epipe',
  'template_name': 'ePIPE-Service-Using-SDP',
  'fill_values': {'service-name': 'Epipe-VLAN-200-DC-East-to-DC-West',
   'customer-id': 500,
   'ne-service-id': 3001,
   'mtu': 9000,
   'site-a.device-id': '10.10.1.1',
   'site-a.endpoint[0].port-id': '1/1/c3/1',
   'site-a.endpoint[0].outer-vlan-tag': 200,
   'site-b.device-id': '10.10.2.2',
   'site-b.endpoint[0].port-id': '1/2/c6/1',
   'site-b.endpoint[0].outer-vlan-tag': 200,
   'sdp[0].sdp-id': '1122',
   'sdp[0].source-device-id': '10.10.1.1',
   'sdp[0].destination-device-id': '10.10.2.2',
   'sdp[1].sdp-id': '2211',
   'sdp[1].source-device-id': '10.10.2.2',
   'sdp[1].destination-device-id': '10.10.1.1'}},
 {'nsp-service-intent:intent': [{'intent-specific-data': {'epipe:epipe': {'admin-state': 'unlocked',
      'customer-id': 500,
      'mtu': 9000,
      'ne-service-id': 3001,
      'sdp-details': {'sdp': [{'admin-state': 'unlocked',
         'destination-device-id': '10.10.2.2',
         'override-v

### 4.3 ePIPE - Formal Business Style

In [7]:
show_example(
    "ePIPE - Formal Business Request",
    "Our customer (ID: 42) needs a dedicated Layer 2 connection between their "
    "172.16.0.10 and 172.16.0.20 routers. Please create an E-Pipe service named "
    "'Epipe-VLAN-500-campus-to-datacenter' with service ID 5500. Use VLAN 500 on ports "
    "1/2/c1/1 (site A) and 1/2/c2/1 (site B). MTU should be 1500. "
    "The SDP tunnel IDs are 1020 and 2010."
)

### ePIPE - Formal Business Request

**User Instruction:**
> Our customer (ID: 42) needs a dedicated Layer 2 connection between their 172.16.0.10 and 172.16.0.20 routers. Please create an E-Pipe service named 'Epipe-VLAN-500-campus-to-datacenter' with service ID 5500. Use VLAN 500 on ports 1/2/c1/1 (site A) and 1/2/c2/1 (site B). MTU should be 1500. The SDP tunnel IDs are 1020 and 2010.

**Step 1: Model extracts fill-values:**

{
  "intent_type": "epipe",
  "template_name": "epipe-service-using-sdp",
  "fill_values": {
    "service-name": "Epipe-VLAN-500-campus-to-datacenter",
    "customer-id": 42,
    "ne-service-id": 5500,
    "mtu": 1500,
    "site-a.device-id": "172.16.0.10",
    "site-a.endpoint[0].port-id": "1/2/c1/1",
    "site-a.endpoint[0].outer-vlan-tag": 500,
    "site-b.device-id": "172.16.0.20",
    "site-b.endpoint[0].port-id": "1/2/c2/1",
    "site-b.endpoint[0].outer-vlan-tag": 500,
    "sdp[0].sdp-id": "1020",
    "sdp[0].source-device-id": "172.16.0.10",
    "sdp[0].destination-device-id": "172.16.0.20",
    "sdp[1].sdp-id": "2010",
    "sdp[1].source-device-id": "172.16.0.20",
    "sdp[1].destination-device-id": "172.16.0.10"
  }
}


**Step 2: Merged into API-ready JSON:**

{
  "nsp-service-intent:intent": [
    {
      "intent-specific-data": {
        "epipe:epipe": {
          "admin-state": "unlocked",
          "customer-id": 42,
          "mtu": 1500,
          "ne-service-id": 5500,
          "sdp-details": {
            "sdp": [
              {
                "admin-state": "unlocked",
                "destination-device-id": "172.16.0.20",
                "override-vc-id": false,
                "sdp-id": "1020",
                "source-device-id": "172.16.0.10",
                "vc-type": "ether"
              },
              {
                "admin-state": "unlocked",
                "destination-device-id": "172.16.0.10",
                "override-vc-id": false,
                "sdp-id": "2010",
                "source-device-id": "172.16.0.20",
                "vc-type": "ether"
              }
            ]
          },
          "site-a": {
            "device-id": "172.16.0.10",
            "endpoint": [
              {
                

({'intent_type': 'epipe',
  'template_name': 'epipe-service-using-sdp',
  'fill_values': {'service-name': 'Epipe-VLAN-500-campus-to-datacenter',
   'customer-id': 42,
   'ne-service-id': 5500,
   'mtu': 1500,
   'site-a.device-id': '172.16.0.10',
   'site-a.endpoint[0].port-id': '1/2/c1/1',
   'site-a.endpoint[0].outer-vlan-tag': 500,
   'site-b.device-id': '172.16.0.20',
   'site-b.endpoint[0].port-id': '1/2/c2/1',
   'site-b.endpoint[0].outer-vlan-tag': 500,
   'sdp[0].sdp-id': '1020',
   'sdp[0].source-device-id': '172.16.0.10',
   'sdp[0].destination-device-id': '172.16.0.20',
   'sdp[1].sdp-id': '2010',
   'sdp[1].source-device-id': '172.16.0.20',
   'sdp[1].destination-device-id': '172.16.0.10'}},
 {'nsp-service-intent:intent': [{'intent-specific-data': {'epipe:epipe': {'admin-state': 'unlocked',
      'customer-id': 42,
      'mtu': 1500,
      'ne-service-id': 5500,
      'sdp-details': {'sdp': [{'admin-state': 'unlocked',
         'destination-device-id': '172.16.0.20',
      

### 4.4 VPRN L3 VPN - Single Site

In [8]:
show_example(
    "VPRN L3 VPN - Single Site with 2 Interfaces",
    "Create a VPRN L3 VPN service named 'VPRN-2000-CloudX' for customer 88. "
    "Configure site on device 192.168.1.10 with service ID 2000. "
    "Route distinguisher: 65001:2000. VRF policies: import CloudX-VRF-Import, export CloudX-VRF-Export. "
    "Interfaces: Alpha-Cluster-Master on port lag-Nokia3-4x10GE with IP 10.100.1.1/31; "
    "Alpha-Cluster-Worker-1 on port 1/1/c5/1 with IP 10.100.1.3/31."
)

### VPRN L3 VPN - Single Site with 2 Interfaces

**User Instruction:**
> Create a VPRN L3 VPN service named 'VPRN-2000-CloudX' for customer 88. Configure site on device 192.168.1.10 with service ID 2000. Route distinguisher: 65001:2000. VRF policies: import CloudX-VRF-Import, export CloudX-VRF-Export. Interfaces: Alpha-Cluster-Master on port lag-Nokia3-4x10GE with IP 10.100.1.1/31; Alpha-Cluster-Worker-1 on port 1/1/c5/1 with IP 10.100.1.3/31.

**Step 1: Model extracts fill-values:**

{
  "intent_type": "vprn",
  "template_name": "VPRNServiceTemplate",
  "fill_values": {
    "service-name": "VPRN-2000-CloudX",
    "customer-id": 88,
    "site[0].site-name": "VPRN-2000-CloudX",
    "site[0].device-id": "192.168.1.10",
    "site[0].ne-service-id": 2000,
    "site[0].route-distinguisher": "65001:2000",
    "site[0].vrf-import": [
      "CloudX-VRF-Import"
    ],
    "site[0].vrf-export": [
      "CloudX-VRF-Export"
    ],
    "site[0].interface[0].interface-name": "Alpha-Cluster-Master",
    "site[0].interface[0].sap.port-id": "lag-Nokia3-4x10GE",
    "site[0].interface[0].ipv4.primary.address": "10.100.1.1",
    "site[0].interface[0].ipv4.primary.prefix-length": 31,
    "site[0].interface[1].interface-name": "Alpha-Cluster-Worker-1",
    "site[0].interface[1].sap.port-id": "1/1/c5/1",
    "site[0].interface[1].ipv4.primary.address": "10.100.1.3",
    "site[0].interface[1].ipv4.primary.prefix-length": 31
  }
}


**Step 2: Merged into API-ready JSON:**

{
  "nsp-service-intent:intent": [
    {
      "service-name": "VPRN-2000-CloudX",
      "intent-type": "vprn",
      "intent-type-version": "2",
      "olc-state": "deployed",
      "template-name": "VPRNServiceTemplate",
      "intent-specific-data": {
        "vprn:vprn": {
          "admin-state": "unlocked",
          "customer-id": 88,
          "sdp-details": {},
          "site-details": {
            "site": [
              {
                "route-aggregation": {},
                "site-name": "VPRN-2000-CloudX",
                "enable-rip": false,
                "route-distinguisher-type": "type0",
                "bgp-evpn": {
                  "enable-evpn-mpls": false
                },
                "interface-details": {
                  "interface": [
                    {
                      "sap": {
                        "enable-filter": false,
                        "cpu-protection": {},
                        "outer-vlan-tag": 0,
                        

({'intent_type': 'vprn',
  'template_name': 'VPRNServiceTemplate',
  'fill_values': {'service-name': 'VPRN-2000-CloudX',
   'customer-id': 88,
   'site[0].site-name': 'VPRN-2000-CloudX',
   'site[0].device-id': '192.168.1.10',
   'site[0].ne-service-id': 2000,
   'site[0].route-distinguisher': '65001:2000',
   'site[0].vrf-import': ['CloudX-VRF-Import'],
   'site[0].vrf-export': ['CloudX-VRF-Export'],
   'site[0].interface[0].interface-name': 'Alpha-Cluster-Master',
   'site[0].interface[0].sap.port-id': 'lag-Nokia3-4x10GE',
   'site[0].interface[0].ipv4.primary.address': '10.100.1.1',
   'site[0].interface[0].ipv4.primary.prefix-length': 31,
   'site[0].interface[1].interface-name': 'Alpha-Cluster-Worker-1',
   'site[0].interface[1].sap.port-id': '1/1/c5/1',
   'site[0].interface[1].ipv4.primary.address': '10.100.1.3',
   'site[0].interface[1].ipv4.primary.prefix-length': 31}},
 {'nsp-service-intent:intent': [{'service-name': 'VPRN-2000-CloudX',
    'intent-type': 'vprn',
    'intent-

### 4.5 VPRN L3 VPN - Multi-Site (2 Sites)

In [9]:
show_example(
    "VPRN L3 VPN - Two Sites",
    "Deploy a multi-site VPRN 'VPRN-3000-EdgeNet' for customer 150. "
    "First site: 192.168.10.1, NE SvcID 3000, RD 65010:3000, "
    "VRF import EdgeNet-VRF-Import, export EdgeNet-VRF-Export. "
    "Interfaces: Gamma-Cluster-Master on port lag-Arista1-2x25GE with IP 10.50.0.1/30; "
    "Gamma-Cluster-Worker-1 on port 1/2/c8/1 with IP 10.50.0.5/30. "
    "Second site: 192.168.10.2, NE SvcID 3000, RD 65020:3000, "
    "VRF import EdgeNet-VRF-Import, export EdgeNet-VRF-Export. "
    "Interfaces: Omega-Cluster-Master on port lag-Dell-NAR-1-4x10GE with IP 10.50.1.1/30."
)

### VPRN L3 VPN - Two Sites

**User Instruction:**
> Deploy a multi-site VPRN 'VPRN-3000-EdgeNet' for customer 150. First site: 192.168.10.1, NE SvcID 3000, RD 65010:3000, VRF import EdgeNet-VRF-Import, export EdgeNet-VRF-Export. Interfaces: Gamma-Cluster-Master on port lag-Arista1-2x25GE with IP 10.50.0.1/30; Gamma-Cluster-Worker-1 on port 1/2/c8/1 with IP 10.50.0.5/30. Second site: 192.168.10.2, NE SvcID 3000, RD 65020:3000, VRF import EdgeNet-VRF-Import, export EdgeNet-VRF-Export. Interfaces: Omega-Cluster-Master on port lag-Dell-NAR-1-4x10GE with IP 10.50.1.1/30.

**Step 1: Model extracts fill-values:**

{
  "intent_type": "vprn",
  "template_name": "VPRN-Service-Template",
  "fill_values": {
    "service-name": "VPRN-3000-EdgeNet",
    "customer-id": 150,
    "site[0].site-name": "VPRN-3000-EdgeNet",
    "site[0].device-id": "192.168.10.1",
    "site[0].ne-service-id": 3000,
    "site[0].route-distinguisher": "65010:3000",
    "site[0].vrf-import": [
      "EdgeNet-VRF-Import"
    ],
    "site[0].vrf-export": [
      "EdgeNet-VRF-Export"
    ],
    "site[0].interface[0].interface-name": "Gamma-Cluster-Master",
    "site[0].interface[0].sap.port-id": "lag-Arista1-2x25GE",
    "site[0].interface[0].ipv4.primary.address": "10.50.0.1",
    "site[0].interface[0].ipv4.primary.prefix-length": 30,
    "site[0].interface[1].interface-name": "Gamma-Cluster-Worker-1",
    "site[0].interface[1].sap.port-id": "1/2/c8/1",
    "site[0].interface[1].ipv4.primary.address": "10.50.0.5",
    "site[0].interface[1].ipv4.primary.prefix-length": 30,
    "site[1].site-name": "VPRN-3000-EdgeNet",
    "site[1]

**Step 2: Merged into API-ready JSON:**

{
  "nsp-service-intent:intent": [
    {
      "service-name": "VPRN-3000-EdgeNet",
      "intent-type": "vprn",
      "intent-type-version": "2",
      "olc-state": "deployed",
      "template-name": "VPRNServiceTemplate",
      "intent-specific-data": {
        "vprn:vprn": {
          "admin-state": "unlocked",
          "customer-id": 150,
          "sdp-details": {},
          "site-details": {
            "site": [
              {
                "route-aggregation": {},
                "site-name": "VPRN-3000-EdgeNet",
                "enable-rip": false,
                "route-distinguisher-type": "type0",
                "bgp-evpn": {
                  "enable-evpn-mpls": false
                },
                "interface-details": {
                  "interface": [
                    {
                      "sap": {
                        "enable-filter": false,
                        "cpu-protection": {},
                        "outer-vlan-tag": 0,
                     

({'intent_type': 'vprn',
  'template_name': 'VPRN-Service-Template',
  'fill_values': {'service-name': 'VPRN-3000-EdgeNet',
   'customer-id': 150,
   'site[0].site-name': 'VPRN-3000-EdgeNet',
   'site[0].device-id': '192.168.10.1',
   'site[0].ne-service-id': 3000,
   'site[0].route-distinguisher': '65010:3000',
   'site[0].vrf-import': ['EdgeNet-VRF-Import'],
   'site[0].vrf-export': ['EdgeNet-VRF-Export'],
   'site[0].interface[0].interface-name': 'Gamma-Cluster-Master',
   'site[0].interface[0].sap.port-id': 'lag-Arista1-2x25GE',
   'site[0].interface[0].ipv4.primary.address': '10.50.0.1',
   'site[0].interface[0].ipv4.primary.prefix-length': 30,
   'site[0].interface[1].interface-name': 'Gamma-Cluster-Worker-1',
   'site[0].interface[1].sap.port-id': '1/2/c8/1',
   'site[0].interface[1].ipv4.primary.address': '10.50.0.5',
   'site[0].interface[1].ipv4.primary.prefix-length': 30,
   'site[1].site-name': 'VPRN-3000-EdgeNet',
   'site[1].device-id': '192.168.10.2',
   'site[1].ne-serv

---
## 5. Summary

| Capability | Status |
|---|---|
| Tunnel intent generation | Validated against real data |
| ePIPE intent generation | Validated against real data |
| VPRN single-site generation | Demonstrated |
| VPRN multi-site generation | Demonstrated |
| SDP bidirectionality | Correctly learned |
| Multiple instruction styles | Formal, conversational, terse all work |
| API-ready JSON output | Directly deployable to NSP RESTConf API |